# Phase 0 smoke test

Verifies that the environment, GPU, and `diffusers` stack can produce a single SD 1.5 text-to-image at 512×512.

Acceptance: runs end-to-end without error; an image is displayed inline.

In [ ]:
import torch

import upscaler

print(f"upscaler {upscaler.__version__}")
print(f"torch {torch.__version__}, cuda_available={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device: {torch.cuda.get_device_name(0)}")

In [ ]:
from diffusers import StableDiffusionPipeline

# stable-diffusion-v1-5/stable-diffusion-v1-5 is the maintained mirror after
# runwayml removed the original repo in 2024.
model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
device = "cuda" if torch.cuda.is_available() else "cpu"

pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=dtype, safety_checker=None)
pipe = pipe.to(device)
pipe.enable_attention_slicing()  # keeps peak VRAM comfortable on 8 GB

In [ ]:
prompt = "a high-resolution photograph of a misty mountain valley at sunrise, sharp detail"
generator = torch.Generator(device=device).manual_seed(0)
image = pipe(prompt, height=512, width=512, num_inference_steps=20, generator=generator).images[0]
image